In [29]:
import re, copy
import requests
import json 

In [30]:
# Function to get game data
API_KEY = "C45399540985DC199A1C374C8EF89F6F"
GENRES_COUNT = 3
CATEGORIES_COUNT = 5

def GetGameInfo(appid):
    url = f"https://store.steampowered.com/api/appdetails?appids={appid}&cc=us"
    data = requests.get(url).json()
    game = data[str(appid)]["data"]
    result = {}
    result["name"] = game["name"]
    result["publisher"] = game["publishers"][0]
    result["genres"] = game["genres"]
    result["platforms"] = game["platforms"]
    result["categories"] = game["categories"]
    result["year"] = game["release_date"]["date"][-4:]
    price_overview = game.get("price_overview")
    if price_overview:
        result["price"] = game["price_overview"]["final"]
    else:
        result["price"] = 0
    return result

In [54]:
# Get Data
METADATA = {}
METADATA[1306630] = GetGameInfo(1306630)
METADATA[537110] = GetGameInfo(537110)
METADATA[2230650] = GetGameInfo(2230650)
METADATA[3887700] = GetGameInfo(3887700)
METADATA[648100] = GetGameInfo(648100)
METADATA[1747760] = GetGameInfo(1747760)
METADATA[730] = GetGameInfo(730)
METADATA[1808500] = GetGameInfo(1808500)
METADATA[1174180] = GetGameInfo(1174180)
METADATA[2807960] = GetGameInfo(2807960)
METADATA[3809440] = GetGameInfo(3809440)
METADATA[359550] = GetGameInfo(359550)

for id in METADATA:
    print(METADATA[id])

{'name': 'Lost Ruins', 'publisher': 'DANGEN Entertainment', 'genres': [{'id': '1', 'description': 'Action'}, {'id': '25', 'description': 'Adventure'}, {'id': '23', 'description': 'Indie'}], 'platforms': {'windows': True, 'mac': True, 'linux': True}, 'categories': [{'id': 2, 'description': 'Single-player'}, {'id': 22, 'description': 'Steam Achievements'}, {'id': 28, 'description': 'Full controller support'}, {'id': 55, 'description': 'DualShock Controller Support'}, {'id': 56, 'description': 'DualShock Controller Support'}, {'id': 23, 'description': 'Steam Cloud'}, {'id': 41, 'description': 'Remote Play on Phone'}, {'id': 42, 'description': 'Remote Play on Tablet'}, {'id': 62, 'description': 'Family Sharing'}], 'year': '2021', 'price': 1999}
{'name': 'Angels of Death', 'publisher': 'Vaka Game Magazine', 'genres': [{'id': '25', 'description': 'Adventure'}, {'id': '23', 'description': 'Indie'}, {'id': '3', 'description': 'RPG'}], 'platforms': {'windows': True, 'mac': False, 'linux': False

In [67]:
# Setup for a query
QUERY_SAMPLE = [
    {"ID":1306630, "Hours":11.8, "Installed":False},
    {"ID":537110, "Hours":6.6, "Installed":False},
    {"ID":2230650, "Hours":93.6, "Installed":True},
    {"ID":3887700, "Hours":52, "Installed":False},
    {"ID":648100, "Hours":35.4, "Installed":False},
]

# Baseline test
total_hours = 0.0
genre_tags = {}
categories = {}
for query in QUERY_SAMPLE:
    total_hours += query["Hours"]
    genre = METADATA[query["ID"]]["genres"]
    for g in genre:
        genre_tags[g["id"]] = 0.0
    category = METADATA[query["ID"]]["categories"]
    for c in category:
        categories[c["id"]] = True
for query in QUERY_SAMPLE:
    genre = METADATA[query["ID"]]["genres"]
    for g in genre:
        genre_tags[g["id"]] += query["Hours"] / len(genre) / total_hours
print(genre_tags)
scores = {}
for id in METADATA:
    genre = METADATA[id]["genres"]
    scores[METADATA[id]["name"]] = 0.0
    for g in genre:
        if g["id"] in genre_tags:
            scores[METADATA[id]["name"]] += genre_tags[g["id"]]
print(scores)

# apply the failure rulescores
scores2 = {}
for id in METADATA:
    category_hit = 0.0
    genre = METADATA[id]["genres"]
    category = METADATA[id]["categories"]
    for c in category:
        if c["id"] in categories:
            category_hit += 1.0
    if category_hit / len(category) <= .5:
        continue
    scores2[METADATA[id]["name"]] = 0.0
    for g in genre:
        if g["id"] in genre_tags:
            scores2[METADATA[id]["name"]] += genre_tags[g["id"]]
print(scores2)

{'1': 0.13707790036776998, '25': 0.23687729856235373, '23': 0.23687729856235373, '3': 0.09796054831160145, '4': 0.11735205616850551, '2': 0.08692744901370777, '37': 0.08692744901370777}
{'Lost Ruins': 0.6108324974924775, 'Angels of Death': 0.571715145436309, 'TEVI': 0.728184553660983, "Girls' Frontline": 0.271815446339017, 'Raging Loop': 0.47375459712470747, 'Momodora: Moonlit Farewell': 0.6108324974924775, 'Counter-Strike 2': 0.22400534938147776, 'ARC Raiders': 0.13707790036776998, 'Red Dead Redemption 2': 0.37395519893012374, 'Battlefield™ 6': 0.13707790036776998, 'Cheese Rolling': 0.578234704112337, "Tom Clancy's Rainbow Six Siege": 0.22400534938147776}
{'Lost Ruins': 0.6108324974924775, 'Angels of Death': 0.571715145436309, 'TEVI': 0.728184553660983, "Girls' Frontline": 0.271815446339017, 'Raging Loop': 0.47375459712470747, 'Momodora: Moonlit Farewell': 0.6108324974924775, 'ARC Raiders': 0.13707790036776998, 'Red Dead Redemption 2': 0.37395519893012374, 'Cheese Rolling': 0.57823470